<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/tiny-transformer/blob/main/Multi_head_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import tensorflow as tf
import numpy as np

# ----------------------------
# 1. Sample English–Tamil data
# ----------------------------
english_sentences = [
    "hello",
    "how are you",
    "i am fine",
    "thank you",
    "good morning"
]

tamil_sentences = [
    "வணக்கம்",
    "நீங்கள் எப்படி இருக்கிறீர்கள்",
    "நான் நன்றாக இருக்கிறேன்",
    "நன்றி",
    "காலை வணக்கம்"
]

tamil_sentences = ["<start> " + s + " <end>" for s in tamil_sentences]

# ----------------------------
# 2. Tokenization
# ----------------------------
def tokenize(sentences):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(filters="")
    tokenizer.fit_on_texts(sentences)
    seq = tokenizer.texts_to_sequences(sentences)
    seq = tf.keras.preprocessing.sequence.pad_sequences(seq, padding="post")
    return seq, tokenizer

en_seq, en_tok = tokenize(english_sentences)
ta_seq, ta_tok = tokenize(tamil_sentences)

# Decoder input & target (teacher forcing)
ta_input = ta_seq[:, :-1]
ta_target = ta_seq[:, 1:]

dataset = tf.data.Dataset.from_tensor_slices(
    ((en_seq, ta_input), ta_target)
).batch(2)

# ----------------------------
# 3. Attention
# ----------------------------
def scaled_dot_product_attention(q, k, v):
    matmul_qk = tf.matmul(q, k, transpose_b=True)
    dk = tf.cast(tf.shape(k)[-1], tf.float32)
    weights = tf.nn.softmax(matmul_qk / tf.math.sqrt(dk), axis=-1)
    return tf.matmul(weights, v)

class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.depth = d_model // num_heads

        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)
        self.dense = tf.keras.layers.Dense(d_model)

    def split(self, x, batch):
        x = tf.reshape(x, (batch, -1, self.num_heads, self.depth))
        return tf.transpose(x, [0, 2, 1, 3])

    def call(self, q, k, v):
        batch = tf.shape(q)[0]

        q = self.split(self.wq(q), batch)
        k = self.split(self.wk(k), batch)
        v = self.split(self.wv(v), batch)

        attn = scaled_dot_product_attention(q, k, v)
        attn = tf.transpose(attn, [0, 2, 1, 3])
        return self.dense(tf.reshape(attn, (batch, -1, self.num_heads * self.depth)))

# ----------------------------
# 4. Encoder & Decoder Layers
# ----------------------------
class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, heads, dff):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, heads)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation="relu"),
            tf.keras.layers.Dense(d_model)
        ])
        self.norm1 = tf.keras.layers.LayerNormalization()
        self.norm2 = tf.keras.layers.LayerNormalization()

    def call(self, x):
        attn = self.mha(x, x, x)
        x = self.norm1(x + attn)
        return self.norm2(x + self.ffn(x))

class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, heads, dff):
        super().__init__()
        self.mha1 = MultiHeadAttention(d_model, heads)
        self.mha2 = MultiHeadAttention(d_model, heads)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation="relu"),
            tf.keras.layers.Dense(d_model)
        ])
        self.norm1 = tf.keras.layers.LayerNormalization()
        self.norm2 = tf.keras.layers.LayerNormalization()
        self.norm3 = tf.keras.layers.LayerNormalization()

    def call(self, x, enc):
        x = self.norm1(x + self.mha1(x, x, x))
        x = self.norm2(x + self.mha2(x, enc, enc))
        return self.norm3(x + self.ffn(x))

# ----------------------------
# 5. Transformer Model (FIXED)
# ----------------------------
class Transformer(tf.keras.Model):
    def __init__(self, vin, vout, d_model=128, heads=4, dff=512):
        super().__init__()
        self.enc_emb = tf.keras.layers.Embedding(vin, d_model)
        self.dec_emb = tf.keras.layers.Embedding(vout, d_model)
        self.encoder = EncoderLayer(d_model, heads, dff)
        self.decoder = DecoderLayer(d_model, heads, dff)
        self.fc = tf.keras.layers.Dense(vout)

    def call(self, inputs):
        inp, tar = inputs   # ✅ SINGLE input tuple
        enc = self.encoder(self.enc_emb(inp))
        dec = self.decoder(self.dec_emb(tar), enc)
        return self.fc(dec)

# ----------------------------
# 6. Train
# ----------------------------
model = Transformer(
    len(en_tok.word_index) + 1,
    len(ta_tok.word_index) + 1
)

model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)

model.fit(dataset, epochs=50)

print("✅ Training complete")

Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - loss: 3.2509
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 1.0966
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.5635
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.2477
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.1494
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0684
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0369
Epoch 8/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0242
Epoch 9/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0164
Epoch 10/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0108
Epoch 11/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0072
Epoch 12/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0051
Epoch 13/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0039
Epoch 14/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0031
Epoch 15/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0026
Epoch 16/50
3/3 ━━━━━━━━━━━━━━━━━

In [3]:
def translate(sentence, max_len=20):
    # Tokenize input sentence
    seq = en_tok.texts_to_sequences([sentence])
    seq = tf.keras.preprocessing.sequence.pad_sequences(
        seq, maxlen=en_seq.shape[1], padding="post"
    )

    # Start token
    start_token = ta_tok.word_index["<start>"]
    end_token = ta_tok.word_index["<end>"]

    output = [start_token]

    for _ in range(max_len):
        dec_input = tf.expand_dims(output, axis=0)

        predictions = model((seq, dec_input), training=False)
        next_token = tf.argmax(predictions[:, -1, :], axis=-1).numpy()[0]

        if next_token == end_token:
            break

        output.append(next_token)

    # Convert tokens to words
    words = [
        ta_tok.index_word.get(i, "")
        for i in output
        if i not in [start_token, end_token]
    ]

    return " ".join(words)

In [12]:
print(translate("how are you"))
print(translate("thank you"))
print(translate("good morning"))

நீங்கள் எப்படி இருக்கிறீர்கள்
நன்றி
காலை வணக்கம்
